In [2]:
import pandas as pd
import os
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [3]:
# load data for evaluation

def load_responses(folder):
    df_dict = {}
    for model in sorted(os.listdir(folder)):
        print(f'Loading responses for {model}...')
        df_dict[model] = {}
        for templ in sorted(os.listdir(folder + '/' + model)):
            if templ.endswith('.csv'):
                df_dict[model][templ.split(".")[0]] = pd.read_csv(folder + '/' + model + '/' + templ)
                assert df_dict[model][templ.split(".")[0]].shape[0] == 500, f"Expected 500 responses for {model}/{templ}, found {df_dict[model][templ.split('.')[0]].shape[0]}"
        #assert len(df_dict[model]) == 8, f"Expected 8 templates for {model}, found {len(df_dict[model])}"
    
    print(f'Loaded {len(df_dict)} models')
    print(f'Loaded {sum([len(df_dict[model]) for model in df_dict])} templates')
    return df_dict

df_dict = load_responses("./responses")
        

Loading responses for Llama-3.1-70B-Instruct...
Loading responses for Llama-3.1-8B-Instruct...
Loading responses for Llama-3.2-3B-Instruct...
Loading responses for Ministral-8B-Instruct-2410...
Loading responses for Mistral-7B-Instruct-v0.3...
Loading responses for Mistral-Nemo-Instruct-2407...
Loading responses for Qwen2.5-72B-Instruct...
Loading responses for chatgpt-4o-latest...
Loading responses for claude-sonnet-4...
Loading responses for deepseek-chat-v3-0324...
Loading responses for gemini-2.0-flash-001...
Loading responses for gemini-2.5-flash...
Loading responses for gemma-2-27b-it...
Loading responses for gemma-2-9b-it...
Loading responses for gpt-3.5-turbo...
Loading responses for gpt-4o-2024-05-13...
Loading responses for gpt-4o-2024-08-06...
Loading responses for gpt-4o-mini-2024-07-18...
Loading responses for llama-3.1-405b-instruct...
Loading responses for llama-3.3-70b-instruct...
Loading responses for llama-4-maverick...
Loaded 21 models
Loaded 168 templates


In [4]:
# parse the eval_text column

def parse_eval_text(eval_text):
    for i in range(1, 6):
        if f"{i}" in str(eval_text):
            return str(i)
    if "refusal" in str(eval_text).lower():
        return "refusal"
    else:
        return "PARSE ERROR"


def parse_dict(df_dict):
    for model in df_dict:
        for templ in df_dict[model]:
            df_dict[model][templ]['eval_label'] = df_dict[model][templ]['eval_text'].apply(parse_eval_text)
            df_dict[model][templ]['final_label'] = df_dict[model][templ]['final_label'].apply(parse_eval_text)

            if len(df_dict[model][templ][df_dict[model][templ]['eval_label'] == 'PARSE ERROR']) > 0:
                print("#" * 80)
                print(f"{model} {templ}: {len(df_dict[model][templ][df_dict[model][templ]['eval_label'] == 'PARSE ERROR'])} PARSE ERRORS")
                #for _, row in df_dict[model][templ][df_dict[model][templ]['eval_label'] == 'PARSE ERROR'].iterrows():
                #    print(row['eval_text'])
                #    print()

    return df_dict

df_dict = parse_dict(df_dict)

################################################################################
Llama-3.2-3B-Instruct templ-1: 1 PARSE ERRORS
################################################################################
Llama-3.2-3B-Instruct templ-2: 1 PARSE ERRORS
################################################################################
Llama-3.2-3B-Instruct templ-4: 1 PARSE ERRORS
################################################################################
Llama-3.2-3B-Instruct templ-5: 6 PARSE ERRORS
################################################################################
Llama-3.2-3B-Instruct templ-6: 2 PARSE ERRORS
################################################################################
Llama-3.2-3B-Instruct templ-7: 15 PARSE ERRORS
################################################################################
Llama-3.2-3B-Instruct templ-8: 3 PARSE ERRORS
################################################################################
Ministral-8B-Instruct-2410 te

In [5]:
# create classification report and confusion matrix

def model_performance_detail(df_dict, model, templ, final_label='final_label', eval_label='eval_label'):
    print("###"*20)
    print(f'#  {model} : {templ}')
    print("###"*20, end='\n\n')
    print(classification_report(df_dict[model][templ][final_label], df_dict[model][templ][eval_label]))
    print(pd.crosstab(df_dict[model][templ][final_label], df_dict[model][templ][eval_label]))
    print()

model_performance_detail(df_dict, "Llama-3.1-70B-Instruct", "templ-5", final_label='final_label', eval_label='eval_label')

#for model in df_dict:
#    for templ in sorted(df_dict[model]):
#        performance_detail(df_dict, model, templ)


############################################################
#  Llama-3.1-70B-Instruct : templ-5
############################################################

              precision    recall  f1-score   support

           1       0.95      0.76      0.84       137
           2       0.49      0.78      0.60        63
           3       0.83      0.58      0.68        93
           4       0.56      0.75      0.64        56
           5       0.86      0.88      0.87        91
     refusal       1.00      0.97      0.98        60

    accuracy                           0.77       500
   macro avg       0.78      0.79      0.77       500
weighted avg       0.82      0.77      0.78       500

eval_label     1   2   3   4   5  refusal
final_label                              
1            104  32   0   1   0        0
2              6  49   6   1   1        0
3              0  17  54  21   1        0
4              0   0   4  42  10        0
5              0   0   1  10  80        0
refu

In [79]:
def model_performance_overview(df_dict, model, final_label='final_label', eval_label='eval_label'):
    overview_df = pd.DataFrame(columns=['accuracy', 'macro_f1', 'weighted_f1'])

    for templ in sorted(df_dict[model]):
        accuracy = accuracy_score(df_dict[model][templ][final_label], df_dict[model][templ][eval_label])
        macro_f1 = round(f1_score(df_dict[model][templ][final_label], df_dict[model][templ][eval_label], average='macro'), 4)
        weighted_f1 = round(f1_score(df_dict[model][templ][final_label], df_dict[model][templ][eval_label], average='weighted'), 4)

        overview_df.loc[templ] = [accuracy, macro_f1, weighted_f1]

    print("###"*20)
    print(f'#  {model} : PERFORMANCE ACROSS TEMPLATES')
    print("###"*20, end='\n\n')
    print("ORIGINAL LABELS: 1, 2, 3, 4, 5, refusal\n")
    display(overview_df)


model_performance_overview(df_dict, "Llama-3.1-70B-Instruct", final_label='final_label', eval_label='eval_label')


############################################################
#  Llama-3.1-70B-Instruct : PERFORMANCE ACROSS TEMPLATES
############################################################

ORIGINAL LABELS: 1, 2, 3, 4, 5, refusal



,accuracy,macro_f1,weighted_f1
templ-1,0.756,0.7351,0.7604
templ-2,0.754,0.7390,0.7625
templ-3,0.654,0.6636,0.6704
templ-4,0.604,0.6180,0.6046
templ-5,0.774,0.7707,0.7821
templ-6,0.776,0.7617,0.7786
templ-7,0.756,0.7565,0.7673
templ-8,0.766,0.7660,0.7762


In [80]:
def full_performance_overview(df_dict, final_label='final_label', eval_label='eval_label', latex=False):
    overview_full_df = pd.DataFrame(columns=sorted(df_dict.keys()))

    for model in df_dict:
        for templ in sorted(df_dict[model]):
            macro_f1 = round(f1_score(df_dict[model][templ][final_label], df_dict[model][templ][eval_label], average='macro'), 4)
            overview_full_df.loc[templ, model] = macro_f1

    # transpose
    overview_full_df = overview_full_df.T

    # add column that is the average across all templates
    overview_full_df['AVERAGE'] = overview_full_df.mean(axis=1)
    overview_full_df = overview_full_df.sort_values(by='AVERAGE', ascending=False)

    print("ORIGINAL LABELS: 1, 2, 3, 4, 5, refusal")
    print("METRIC: MACRO F1 SCORE\n")
    display(overview_full_df.style.format("{:.2}"))

    if latex:
        # iterate through rows of df, and print each row for the latex table. the row should end with "\\" and not "&"
        # for every second row, start with "\rowcolor[HTML]{F0F0F0} "
        for model in overview_full_df.index:
            print(f'{model} & ', end='')
            for val in overview_full_df.loc[model]:
                if val == overview_full_df.loc[model, 'AVERAGE']:
                    print(f'{val:.2f}', end=' \\\\ \n')
                else:
                    print(f'{val:.2f} & ', end='')

full_performance_overview(df_dict, final_label='final_label', eval_label='eval_label', latex=True)

ORIGINAL LABELS: 1, 2, 3, 4, 5, refusal
METRIC: MACRO F1 SCORE



,templ-1,templ-2,templ-3,templ-4,templ-5,templ-6,templ-7,templ-8,AVERAGE
gemini-2.5-flash,0.79,0.75,0.7,0.77,0.82,0.82,0.82,0.81,0.79
chatgpt-4o-latest,0.77,0.75,0.71,0.7,0.81,0.8,0.81,0.82,0.77
gemini-2.0-flash-001,0.76,0.78,0.68,0.7,0.78,0.79,0.78,0.77,0.76
claude-sonnet-4,0.78,0.78,0.61,0.71,0.79,0.79,0.67,0.77,0.74
Llama-3.1-70B-Instruct,0.74,0.74,0.66,0.62,0.77,0.76,0.76,0.77,0.73
Qwen2.5-72B-Instruct,0.69,0.71,0.6,0.67,0.76,0.74,0.74,0.76,0.71
gpt-4o-2024-05-13,0.73,0.72,0.62,0.62,0.73,0.71,0.74,0.75,0.7
gpt-4o-mini-2024-07-18,0.66,0.71,0.69,0.65,0.72,0.69,0.72,0.71,0.69
gpt-4o-2024-08-06,0.7,0.69,0.6,0.64,0.72,0.71,0.73,0.73,0.69
llama-3.1-405b-instruct,0.73,0.64,0.57,0.61,0.68,0.67,0.66,0.67,0.65


gemini-2.5-flash & 0.79 & 0.75 & 0.70 & 0.77 & 0.82 & 0.82 & 0.82 & 0.81 & 0.79 \\ 
chatgpt-4o-latest & 0.77 & 0.75 & 0.71 & 0.70 & 0.81 & 0.80 & 0.81 & 0.82 & 0.77 \\ 
gemini-2.0-flash-001 & 0.76 & 0.78 & 0.68 & 0.70 & 0.78 & 0.79 & 0.78 & 0.77 & 0.76 \\ 
claude-sonnet-4 & 0.78 & 0.78 & 0.61 & 0.71 & 0.79 & 0.79 & 0.67 & 0.77 & 0.74 \\ 
Llama-3.1-70B-Instruct & 0.74 & 0.74 & 0.66 & 0.62 & 0.77 & 0.76 & 0.76 & 0.77 & 0.73 \\ 
Qwen2.5-72B-Instruct & 0.69 & 0.71 & 0.60 & 0.67 & 0.76 & 0.74 & 0.74 & 0.76 & 0.71 \\ 
gpt-4o-2024-05-13 & 0.73 & 0.72 & 0.62 & 0.62 & 0.73 & 0.71 & 0.74 & 0.75 & 0.70 \\ 
gpt-4o-mini-2024-07-18 & 0.66 & 0.71 & 0.69 & 0.65 & 0.72 & 0.69 & 0.72 & 0.71 & 0.69 \\ 
gpt-4o-2024-08-06 & 0.70 & 0.69 & 0.60 & 0.64 & 0.72 & 0.71 & 0.73 & 0.73 & 0.69 \\ 
llama-3.1-405b-instruct & 0.73 & 0.64 & 0.57 & 0.61 & 0.68 & 0.67 & 0.66 & 0.67 & 0.65 \\ 
llama-3.3-70b-instruct & 0.62 & 0.64 & 0.62 & 0.53 & 0.65 & 0.77 & 0.63 & 0.63 & 0.64 \\ 
Mistral-7B-Instruct-v0.3 & 0.60 & 0.62 & 

In [81]:
# collapse labels: 1 and 2 -> pro, 3 -> neutral, 4 and 5 -> con, refusal -> refusal
# then repeat the classification report and confusion matrix

def collapse_labels(label):
    if label in ['1', '2']:
        return 'pro'
    elif label == '3':
        return 'neutral'
    elif label in ['4', '5']:
        return 'con'
    elif label == 'refusal':
        return 'refusal'
    else:
        return 'ERROR'
    
for model in df_dict:
    
        for templ in sorted(df_dict[model]):
    
            df_dict[model][templ]['eval_label_collapsed'] = df_dict[model][templ]['eval_label'].apply(collapse_labels)
            df_dict[model][templ]['final_label_collapsed'] = df_dict[model][templ]['final_label'].apply(collapse_labels)
    
            print("###"*20)
            print(f'#  {model} : {templ}')
            print("###"*20, end='\n\n')
            print(classification_report(df_dict[model][templ]['final_label_collapsed'], df_dict[model][templ]['eval_label_collapsed']))
            print(pd.crosstab(df_dict[model][templ]['final_label_collapsed'], df_dict[model][templ]['eval_label_collapsed']))
            print()


############################################################
#  Llama-3.1-70B-Instruct : templ-1
############################################################

              precision    recall  f1-score   support

         con       0.84      0.87      0.86       147
     neutral       0.70      0.67      0.68        93
         pro       0.92      0.94      0.93       200
     refusal       1.00      0.88      0.94        60

    accuracy                           0.86       500
   macro avg       0.86      0.84      0.85       500
weighted avg       0.86      0.86      0.86       500

eval_label_collapsed   con  neutral  pro  refusal
final_label_collapsed                            
con                    128       19    0        0
neutral                 18       62   13        0
pro                      3        8  189        0
refusal                  3        0    4       53

############################################################
#  Llama-3.1-70B-Instruct : templ-2
########

/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c



############################################################
#  Mistral-7B-Instruct-v0.3 : templ-6
############################################################

              precision    recall  f1-score   support

         con       0.73      0.99      0.84       147
     neutral       0.75      0.52      0.61        93
         pro       0.91      0.90      0.90       200
     refusal       1.00      0.68      0.81        60

    accuracy                           0.83       500
   macro avg       0.85      0.77      0.79       500
weighted avg       0.84      0.83      0.82       500

eval_label_collapsed   con  neutral  pro  refusal
final_label_collapsed                            
con                    145        1    1        0
neutral                 30       48   15        0
pro                     11       10  179        0
refusal                 12        5    2       41

############################################################
#  Mistral-7B-Instruct-v0.3 : templ-7
##

/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c



############################################################
#  gemini-2.5-flash : templ-3
############################################################

              precision    recall  f1-score   support

         con       0.85      0.76      0.80       147
     neutral       0.60      0.84      0.70        93
         pro       0.93      0.75      0.83       200
     refusal       0.69      0.88      0.77        60

    accuracy                           0.79       500
   macro avg       0.77      0.81      0.78       500
weighted avg       0.82      0.79      0.79       500

eval_label_collapsed   con  neutral  pro  refusal
final_label_collapsed                            
con                    112       22    5        8
neutral                  6       78    4        5
pro                     10       29  150       11
refusal                  4        1    2       53

############################################################
#  gemini-2.5-flash : templ-4
##################

/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

eval_label_collapsed   con  neutral  pro  refusal
final_label_collapsed                            
con                    137       10    0        0
neutral                 11       69   13        0
pro                      0       12  188        0
refusal                  2        0    1       57

############################################################
#  gpt-4o-2024-08-06 : templ-8
############################################################

              precision    recall  f1-score   support

         con       0.90      0.95      0.92       147
     neutral       0.77      0.69      0.73        93
         pro       0.92      0.94      0.93       200
     refusal       1.00      0.93      0.97        60

    accuracy                           0.90       500
   macro avg       0.90      0.88      0.89       500
weighted avg       0.89      0.90      0.89       500

eval_label_collapsed   con  neutral  pro  refusal
final_label_collapsed                            
con       

/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/paul/Documents/Repos/issuebench/env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

In [82]:
for model in df_dict:

    # create overview df for each model: each row is a template, columns are accuracy and macro f1 score and weighted f1 score

    overview_df = pd.DataFrame(columns=['accuracy', 'macro_f1', 'weighted_f1'])

    for templ in sorted(df_dict[model]):
        accuracy = accuracy_score(df_dict[model][templ]['final_label_collapsed'], df_dict[model][templ]['eval_label_collapsed'])
        macro_f1 = round(f1_score(df_dict[model][templ]['final_label_collapsed'], df_dict[model][templ]['eval_label_collapsed'], average='macro'),4)
        weighted_f1 = round(f1_score(df_dict[model][templ]['final_label_collapsed'], df_dict[model][templ]['eval_label_collapsed'], average='weighted'),4)

        overview_df.loc[templ] = [accuracy, macro_f1, weighted_f1]

    print("###"*20)
    print(f'#  {model} : OVERVIEW')
    print("###"*20, end='\n\n')
    print("COLLAPSED LABELS: 1,2 -> pro, 3 -> neutral, 4,5 -> con, refusal -> refusal\n")
    print(overview_df, end='\n\n')



############################################################
#  Llama-3.1-70B-Instruct : OVERVIEW
############################################################

COLLAPSED LABELS: 1,2 -> pro, 3 -> neutral, 4,5 -> con, refusal -> refusal

         accuracy  macro_f1  weighted_f1
templ-1     0.864    0.8516       0.8634
templ-2     0.858    0.8503       0.8597
templ-3     0.814    0.8223       0.8172
templ-4     0.766    0.7717       0.7771
templ-5     0.890    0.8755       0.8838
templ-6     0.906    0.8977       0.9031
templ-7     0.876    0.8591       0.8721
templ-8     0.874    0.8517       0.8671

############################################################
#  Llama-3.1-8B-Instruct : OVERVIEW
############################################################

COLLAPSED LABELS: 1,2 -> pro, 3 -> neutral, 4,5 -> con, refusal -> refusal

         accuracy  macro_f1  weighted_f1
templ-1     0.752    0.5853       0.7089
templ-2     0.762    0.7078       0.7565
templ-3     0.696    0.5753       0.

In [83]:
# create full overview: each row is a template, each column is a model, values are macro f1 score
overview_full_df = pd.DataFrame(columns=sorted(df_dict.keys()))

for model in df_dict:
    for templ in sorted(df_dict[model]):
        macro_f1 = round(f1_score(df_dict[model][templ]['final_label_collapsed'], df_dict[model][templ]['eval_label_collapsed'], average='macro'),4)
        overview_full_df.loc[templ, model] = macro_f1

# transpose
overview_full_df = overview_full_df.T

# add column that is the average across all templates
overview_full_df['AVERAGE'] = overview_full_df.mean(axis=1)
overview_full_df = overview_full_df.sort_values(by='AVERAGE', ascending=False)

print("###"*20)
print(f'#  FULL OVERVIEW')
print("###"*20, end='\n\n')
print("COLLAPSED LABELS: 1,2 -> pro, 3 -> neutral, 4,5 -> con, refusal -> refusal\n")
print("METRIC: MACRO F1 SCORE\n")
display(overview_full_df.style.format("{:.2}"))



############################################################
#  FULL OVERVIEW
############################################################

COLLAPSED LABELS: 1,2 -> pro, 3 -> neutral, 4,5 -> con, refusal -> refusal

METRIC: MACRO F1 SCORE



,templ-1,templ-2,templ-3,templ-4,templ-5,templ-6,templ-7,templ-8,AVERAGE
gemini-2.5-flash,0.88,0.82,0.78,0.84,0.91,0.92,0.9,0.9,0.87
gpt-4o-2024-08-06,0.88,0.86,0.77,0.77,0.9,0.91,0.9,0.89,0.86
gemini-2.0-flash-001,0.86,0.88,0.81,0.78,0.89,0.89,0.88,0.87,0.86
gpt-4o-2024-05-13,0.88,0.87,0.75,0.72,0.9,0.9,0.91,0.9,0.85
chatgpt-4o-latest,0.84,0.83,0.8,0.79,0.89,0.89,0.89,0.88,0.85
Llama-3.1-70B-Instruct,0.85,0.85,0.82,0.77,0.88,0.9,0.86,0.85,0.85
Qwen2.5-72B-Instruct,0.82,0.82,0.77,0.75,0.89,0.89,0.88,0.88,0.84
claude-sonnet-4,0.88,0.89,0.65,0.84,0.89,0.89,0.71,0.88,0.83
gpt-4o-mini-2024-07-18,0.81,0.84,0.84,0.74,0.85,0.85,0.82,0.81,0.82
gemma-2-27b-it,0.67,0.78,0.77,0.73,0.85,0.86,0.82,0.78,0.78
